<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Hybrid search

**[Hybrid search](https://learning.nextia-ai.com/courses/search/m05/hybrid-search/)** · Search and Ranking: Find the Best Match · Module 5, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** fuse a lexical and a dense list with reciprocal rank fusion and with score blending, choose k and alpha on the validation queries, and measure the choice once on the test queries.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 40 MB). No account and no key. |
| **You should know** | [Where dense search fails](https://learning.nextia-ai.com/courses/search/m05/where-dense-search-fails/), and why you tune on validation. |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m05/hybrid-search/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M05-L03-hybrid-search/hybrid-search-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 41 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/embeddings/products.npy": "f422870c4ab75364fad883fa8858bd7f31f2591bad07753a8f713a0872824ab0",
    "data/embeddings/queries.npy": "5333da6bd9ac44ec1bf92ddf916ec60cb557466b90b5075711f59c8e61f96253",
    "data/embeddings/model.json": "b5d1088cde17e0eb1bb9565f2e1e45c8cfb5b6a545ae72e44404d2429d6d9af4",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Reciprocal rank fusion by hand

Two short lists: a lexical one and a dense one. **Reciprocal rank fusion (RRF)** gives each product 1 ÷ (k + rank) from each list that contains it, and adds the parts. Run the next cell.

In [ ]:
def rrf(lists, k=60):
    """Reciprocal rank fusion: each list gives 1 / (k + rank); a product that a list lacks gets nothing from it."""
    total = {}
    for ranked in lists:
        for rank, pid in enumerate(ranked, 1):
            total[pid] = total.get(pid, 0.0) + 1.0 / (k + rank)
    return sorted(total.items(), key=lambda kv: (-kv[1], kv[0]))

lexical = ["A", "B", "C"]
dense = ["C", "D", "A"]
for pid, s in rrf([lexical, dense]):
    print(pid, round(s, 5))

> **Look.** A and C are in both lists, so they come first, with the same sum. B and D are each in one list only.

## 2. Your turn: one RRF score

Compute the RRF score of D with k = 60.

In [ ]:
# A is 1st (lexical) and 3rd (dense); C is 3rd and 1st: the same sum. D is only in the dense list, 2nd.
score_d = 1 / (60 + 2)
print(round(score_d, 5))

In [ ]:
expect('score_d', round(score_d, 5) if score_d is not ... else None, 0.01613)

## 3. Two lists for every query

The next cell builds a plain BM25 (one field: the title, the French title and the description) and keeps both score arrays for every validation and test query. It is simpler than the project's best lexical run, so your numbers differ from the lesson's. Run it (about 30 seconds).

In [ ]:
import re, bm25s
cat = catalogue()
q = queries()
j = judgments()
ids = cat["product_id"].to_numpy()
P = np.load("data/embeddings/products.npy").astype(np.float32)
Q = np.load("data/embeddings/queries.npy").astype(np.float32)

def tokens(text):
    return [t for t in re.split(r"[^\w]+|_", str(text).lower()) if t]

corpus = [tokens(t) for t in cat["title"] + " " + cat["title_fr"] + " " + cat["description"]]
lex = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
lex.index(corpus, show_progress=False)

def top(score, k=100):
    idx = np.argpartition(-score, k)[:k]
    return list(ids[idx[np.lexsort((ids[idx], -score[idx]))]])

def bm25_scores(text):
    words = [w for w in tokens(text) if w in lex.vocab_dict]
    return lex.get_scores(words) if words else np.zeros(len(ids))

# For every validation and test query: both score arrays (cached, so tuning is fast).
qs = q[q["split"].isin(["validation", "test"])]
S_lex = {qid: bm25_scores(text) for qid, text in zip(qs["query_id"], qs["query"])}
S_den = {qid: P @ Q[row] for qid, row in zip(qs["query_id"], qs.index)}
val = q[q["split"] == "validation"]["query_id"].tolist()
test = q[q["split"] == "test"]["query_id"].tolist()

def ndcg(run, which):
    return cp.per_query({qid: run(qid) for qid in which}, j)["ndcg@10"].mean()

bm25_run = lambda qid: top(S_lex[qid])[:10] if S_lex[qid].max() > 0 else []
dense_run = lambda qid: top(S_den[qid])[:10]
print(f"validation: bm25 {ndcg(bm25_run, val):.3f}, dense {ndcg(dense_run, val):.3f}")

> **Check.** On validation: bm25 about 0.533, dense about 0.458.

## 4. Your turn: choose k on validation

Fuse the two top-100 lists of each query with `rrf`, for k = 10, 30, 60 and 100, and measure each on the validation queries only.

In [ ]:
def rrf_run(k):
    return lambda qid: [p for p, _ in rrf([top(S_lex[qid]), top(S_den[qid])], k)][:10]

rrf_val = {k: round(float(ndcg(rrf_run(k), val)), 3) for k in [10, 30, 60, 100]}
print(rrf_val)
best_k = max(rrf_val, key=rrf_val.get)
print("best k on validation:", best_k)

In [ ]:
expect_hash('rrf_val', rrf_val, '05530f559c35ca14')

> **Check.** {10: 0.526, 30: 0.534, 60: 0.534, 100: 0.533}. The best k is 30 (k 60 is as good to 3 decimals); the values are so close that any k from 30 to 100 would do.

## 5. Score blending

**Score blending** scales each list to 0 to 1 and mixes them: alpha × BM25 + (1 − alpha) × dense. Run the next cell; it defines the blend as in the project.

In [ ]:
def blend(qid, alpha):
    """alpha x BM25 (divided by its best score) + (1 - alpha) x dense (0..1 over its top 100)."""
    lx = S_lex[qid] / S_lex[qid].max() if S_lex[qid].max() > 0 else S_lex[qid]
    dn = S_den[qid]
    t = np.partition(dn, -100)[-100:]
    dn = np.clip((dn - t.min()) / (t.max() - t.min()), 0, 1)
    return alpha * lx + (1 - alpha) * dn

def blend_run(alpha):
    return lambda qid: top(blend(qid, alpha))[:10]

## 6. Your turn: choose alpha on validation

In [ ]:
blend_val = {a: round(float(ndcg(blend_run(a), val)), 3) for a in [0.2, 0.4, 0.5, 0.6, 0.8]}
print(blend_val)
best_alpha = max(blend_val, key=blend_val.get)
print("best alpha on validation:", best_alpha)

In [ ]:
expect_hash('blend_val', blend_val, '1d94788ee74595cd')

> **Check.** {0.2: 0.477, 0.4: 0.507, 0.5: 0.529, 0.6: 0.553, 0.8: 0.556}: the best alpha is 0.8. More weight on BM25 is better, as in the lesson.

## 7. Measure the chosen settings once on the test queries

Only now look at the test queries, once, with the settings you chose. Run the next cell: the paired difference against BM25, with its 95% interval.

In [ ]:
if best_k is ... or best_alpha is ...:
    print("Finish the tasks above first.")
else:
    pq = {name: cp.per_query({qid: run(qid) for qid in test}, j)
          for name, run in [("bm25", bm25_run), ("rrf", rrf_run(best_k)), ("blend", blend_run(best_alpha))]}
    for name in ["rrf", "blend"]:
        d = cp.paired(pq["bm25"], pq[name])
        print(f"{name} minus bm25 on {d['queries']} test queries: {d['difference']:+.3f} "
              f"({d['low']:+.3f} to {d['high']:+.3f}); better {d['better']}, worse {d['worse']}")

> **Think.** Is either gain clear (an interval above zero)? Compare with the lesson: there, against a stronger lexical run, RRF gave no gain and the blend a small one.

In the project: `search compare bm25-norm+spell+syn hybrid-rrf --by qtype` and the same for `hybrid-blend`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Many languages](https://learning.nextia-ai.com/courses/search/m05/many-languages/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m05/hybrid-search/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.